In [ ]:
import Sunlimb
import os
import re
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import image
import sunpy.map
from astropy.io import fits
import warnings
warnings.filterwarnings("ignore")
%matplotlib notebook

In [ ]:
path_movie = "E:/python/projects/alfven/data/hrieuvopn/10moving/movie/"
path = "E:/python/projects/alfven/data/hrieuvopn/10moving/"

In [ ]:
files = [os.path.abspath(os.path.join(path, file)) for file in os.listdir(path)]
files = [file for file in files if file.endswith('.fits')]

def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)

In [ ]:
def get_name(filename):
    match = re.search(r'(\d{8}T\d{9})', filename)
    if match:
        return match.group(1)
    else:
        return None

In [ ]:
def atrous_transform(file):
    data, header = fits.getdata(file, header=True)
    coe = image.a_trous(data, 4, method='Triangle')
    transformed_data = coe.data[1]+coe.data[2]
    atrous_map = sunpy.map.Map(transformed_data, header)
    return atrous_map

In [ ]:
def draw_fig(*args):
    file, output_path, ax = args
    ax.clear()
    euv_map = atrous_transform(file)
    data, r_vect, deg_vect = Sunlimb.to_polar(euv_map, (179, 182), r_range=(1, 1.05), d_deg=0.01)
    Sunlimb.plot_data(ax, data, deg_vect, r_vect, xlabel='degree (deg)', ylabel='radius (Rs)', title=euv_map.fits_header['date-obs'])
    y_line = 20 * 1e6 / euv_map.fits_header['RSUN_REF'] + 1
    ax.hlines(y_line, 179, 182, color='k', linestyle='--')
    name = get_name(file) + '.png'
    plt.savefig(output_path+'fig_cut_atrous/'+name, dpi=500)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

for file in tqdm(files):
    draw_fig(file, path_movie, ax)
        
plt.close(fig)

In [ ]:
image.images_to_video(path_movie+'fig_cut_atrous/', path_movie+'movie_cut_atrous.mp4', fps=30)